In [21]:
import pandas as pd
import numpy as np


In [22]:
# 2. Load Dataset
df = pd.read_csv("messy_dataset.csv")


In [23]:
# 2. Remove NaN Target (CRITICAL)

print("Before cleaning:", df.shape)

df = df.dropna(subset=["target"])

print("After cleaning:", df.shape)
print("Remaining NaN in target:", df["target"].isnull().sum())

Before cleaning: (500, 5)
After cleaning: (450, 5)
Remaining NaN in target: 0


In [24]:
# 3. Features & Target

num_cols = df.select_dtypes(include=np.number).columns.tolist()
num_cols.remove("target")

X = df[num_cols].copy()
y = df["target"].copy()

In [25]:
# 4. Outlier Handling

def cap_outliers(df, cols):
    df = df.copy()
    for col in cols:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1

        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR

        df[col] = np.clip(df[col], lower, upper)
    return df

X = cap_outliers(X, num_cols)


In [26]:
# 5. Train-Test Split


from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 🔥 SAFETY CHECK (MUST BE ZERO)
print("NaN in y_train:", y_train.isnull().sum())
print("NaN in y_test:", y_test.isnull().sum())

NaN in y_train: 0
NaN in y_test: 0


In [27]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

In [28]:


pipeline_mean = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler())
])

pipeline_median = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", RobustScaler())
])

pipeline_knn = Pipeline([
    ("imputer", KNNImputer(n_neighbors=5)),
    ("scaler", StandardScaler())
])

preprocessor_mean = ColumnTransformer([
    ("num", pipeline_mean, num_cols)
])

preprocessor_median = ColumnTransformer([
    ("num", pipeline_median, num_cols)
])

preprocessor_knn = ColumnTransformer([
    ("num", pipeline_knn, num_cols)
])

In [29]:
def evaluate(preprocessor):
    model = Pipeline([
        ("prep", preprocessor),
        ("model", LinearRegression())
    ])
    
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    
    return mean_squared_error(y_test, preds)

mse_mean = evaluate(preprocessor_mean)
mse_median = evaluate(preprocessor_median)
mse_knn = evaluate(preprocessor_knn)

print("Mean:", mse_mean)
print("Median:", mse_median)
print("KNN:", mse_knn)

Mean: 2905.1297366080044
Median: 2903.7778262671263
KNN: 2928.8297402466155


In [30]:
results = {
    "Mean": mse_mean,
    "Median": mse_median,
    "KNN": mse_knn
}

best_method = min(results, key=results.get)

print("All Results:", results)
print("\nBest Imputation Strategy:", best_method)
print("Best MSE:", results[best_method])

All Results: {'Mean': 2905.1297366080044, 'Median': 2903.7778262671263, 'KNN': 2928.8297402466155}

Best Imputation Strategy: Median
Best MSE: 2903.7778262671263
